In [29]:
from data import *

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# generate data
d1 = 1000
d2 = 100
r = 1
p = 2.0 / d2

M = get_random_matrix(d1, d2, r) / np.sqrt(d1)
print(M)
observed_M, masks = get_uniformly_random_samples(M, p)
print(d1*d2)
print(p)
print(masks.sum())
print(observed_M)

In [ ]:
#eps = 0.01

cov_observe_M =  observed_M.T @ observed_M
cov_observe_count = (observed_M == 0).T @ (observed_M == 0)
diag_cov = np.diag( np.diag(cov_observe_M) )

np.count_nonzero(cov_observe_M)

In [32]:
cov_observe_count = (1 * (observed_M != 0)).T @ (1 * (observed_M != 0))
cov_observe_count = cov_observe_count + (cov_observe_count == 0) * 1
T = cov_observe_M / (cov_observe_count/d1)

In [ ]:
T_masks = np.nonzero(T)
MTM = M.T @ M

mask_err = T[T_masks] - MTM[T_masks]

print(np.linalg.norm(mask_err) / np.linalg.norm(MTM, 'fro'))

In [ ]:
T_p = (1.0 / p) * diag_cov + (1.0 / (p**2)) * (cov_observe_M - diag_cov)

mask_err_Tp = T_p[T_masks] - MTM[T_masks]

print(np.linalg.norm(mask_err_Tp) / np.linalg.norm(MTM, 'fro'))

In [ ]:
# impute missing values from rank-r SVD corresponding to masks

train_losses = []
err_estimates = []

epochs = 30
#lr = 0.05
X = T
T_masks = 1 * (T != 0)
for i in range(epochs):
    U, D, Vt = np.linalg.svd(X)
    D[r:] = 0
    X_update = U @ np.diag(D) @ Vt

    X = T * T_masks + X_update * (1 - T_masks)

    err = MTM - X
    loss = (err**2).mean()
    train_losses.append(loss)
    relative_err = np.linalg.norm(err, 'fro') / np.linalg.norm(MTM, 'fro') 
    err_estimates.append(relative_err)
    #print(relative_err)

plt.figure(figsize=(5, 3))
plt.plot(train_losses, label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss')
plt.legend()
plt.show()

plt.figure(figsize=(5, 3))
plt.plot(err_estimates, label='Err Estimation')
plt.xlabel('Epoch')
plt.ylabel('Err')
plt.title('Err Estimation')
plt.legend()

In [ ]:
# run soft impute
epochs = 30
lr = 0.05
X = T
for i in range(epochs):
    U, D, Vt = np.linalg.svd(X)
    D[r:] = 0
    X_update = U @ np.diag(D) @ Vt

    #if np.linalg.norm(X - X_update, 'fro') / np.linalg.norm(X) < eps:
    #    print(i)
    #    break

    X = (1 - lr) * X + lr * X_update

    # print distance
    err = M.T @ M - X
    relative_err = np.linalg.norm(err, 'fro') / np.linalg.norm(M.T @ M)
    print(relative_err)

#print(D)

In [ ]:
err = M.T @ M - T
relative_err = np.linalg.norm(err, 'fro') / np.linalg.norm(M.T @ M)
print(relative_err)

In [ ]:
cov_observe_count

In [ ]:
T